**Parallel chains in LangChain allow you to run multiple independent tasks at the same time using RunnableParallel or a standard Python dictionary inside LangChain Expression Language (LCEL)**.

* Key Concepts
    * **Concurrency**: Runs multiple runnables simultaneously with the same input, saving total execution time.
    * **Python Dict Shorthand**: You can pass a dictionary where keys are output names and values are separate chains or runnables.
    * **Output Format**: Returns a dictionary matching your keys, holding the results from each individual branc

In [1]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

## This function will load all the variable from .env file and will make them available in
## os.environ directory (env_variablea)
load_dotenv()

import os 

if os.environ.get("OPENAI_API_KEY"):
    print("✅ OPENAI_API_KEY Exists.")
else:
    raise ValueError("❌ OPENAI_API_KEY Not Found...")


✅ OPENAI_API_KEY Exists.


### Chain with Parallel Chains

In [2]:
# Task 1: Prompt
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a movie Summary generator'),
    ('human', "please summarise the movie in brief: {input}")
])

In [3]:
# Task 2: LLM
llm_openai = ChatOpenAI(model='gpt-5-mini',
                        temperature=0)

In [4]:
# Task 3; Output parser
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [5]:
# task 4: Custom Runnable
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {'text': text}

dict_maker_runnable = RunnableLambda(dictionary_maker)

### Parallel Chain 1

In [6]:
# Task 1: Prompt
linkedin_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Linkedin Post Generator'),
    ('human', 'crearte a post for the following topic: {text}')
])

# Task 2: LLM
llm_openai = ChatOpenAI(model='gpt-5-mini',
                        temperature=0)

# Task 3; Output parser
str_parser = StrOutputParser()

# Linkedin Chain
chain = linkedin_prompt | llm_openai | str_parser

### Parallel Chain 2

In [7]:
def insta_chain(text:dict):
    
    text = text['text']
    
    # Task 1: Prompt
    insta_prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a Instagram Post Generator'),
        ('human', 'crearte a post for the following topic: {text}')
    ])

    # Task 2: LLM
    llm_openai = ChatOpenAI(model='gpt-5-mini',
                            temperature=0)

    # Task 3; Output parser
    str_parser = StrOutputParser()
    
    chain_insta = insta_prompt | llm_openai | str_parser
    result = chain_insta.invoke(text)
    
    return result

insta_chain_runnable = RunnableLambda(insta_chain)

### Final Orchestration

In [8]:
from langchain_core.runnables import RunnableParallel

final_chain = (
                    prompt_template | 
                    llm_openai | 
                    str_parser | 
                    dict_maker_runnable | 
                    RunnableParallel(
                        branches={
                        "linkedin": chain,
                        "instagram": insta_chain_runnable
                    })
)

In [9]:
final_chain.invoke("Swadesh")

{'branches': {'linkedin': "Watching Swades (2004) is more than a film exercise—it's a compact masterclass in purpose-driven leadership.\n\nQuick recap: Mohan Bhargava, a successful Indian-born NASA engineer (Shah Rukh Khan), returns to his village to find his childhood nanny. Confronted with everyday struggles—no electricity, poor infrastructure—he uses his technical skills, empathy, and persistence to design sustainable solutions and, ultimately, chooses to stay and work for grassroots development.\n\nLeadership takeaways for professionals:\n- Purpose trumps pedigree: technical success is amplified when paired with meaning and community impact.\n- Skills are transferable: domain expertise can solve grassroots problems when adapted to local constraints.\n- Design for sustainability: solutions must be maintainable and locally owned to last.\n- Servant leadership works: listening, patience, and trust-building unlock collective action.\n- Personal trade-offs matter: meaningful work often 

### Chain as runnable

In [12]:
# Task 1: Beautify function
def beautify_output(final_response: dict) -> dict:
    
    linkedin_response = final_response['branches']["linkedin"],
    instagram_response = final_response['branches']["instagram"]
    
    return {
        "linkedin": linkedin_response,
        "instagram": instagram_response
    }
    
beautify_runnable = RunnableLambda(beautify_output)

# Task 2: Final Chain
# Beautify Chain
beautify_chain = final_chain | beautify_runnable

beautify_chain.invoke("Tamasha")

{'linkedin': ("Tamasha (2015) — at surface a love story, but at heart a sharp lesson about identity, creativity and the cost of losing ourselves to other people's expectations.\n\nQuick recap: Ved (Ranbir Kapoor) is a wildly imaginative storyteller who’s pushed into a “normal” mold by family pressure. On a trip to Corsica he sheds that persona and connects deeply with Tara (Deepika Padukone). Back home, he reverts to the safe, conventional version of himself; the mismatch between who he is and who he’s expected to be leads to a breakdown — and, ultimately, a difficult but necessary journey back toward authenticity.\n\nWhat this taught me (and what leaders can take into the workplace):\n- Authenticity fuels creativity and long-term engagement; forcing conformity kills both.  \n- Psychological safety matters: people need permission to bring their whole selves to work.  \n- Creativity isn’t a luxury; it’s a capability that requires time, space and encouragement.  \n- Healing and growth of